# categorical encoding

label encoding vs one-hot vs target encoding (manual). use a synthetic dataset with one categorical column.

In [1]:
import pandas as pd
import numpy as np

rng = np.random.RandomState(0)
df = pd.DataFrame({
    'city': rng.choice(['ny','sf','la','chi'], size=200),
    'x': rng.randn(200),
})
df['y'] = (df['x'] + (df['city']=='sf').astype(int) * 1.5 + rng.randn(200) * 0.5 > 0).astype(int)
df.head()

## one-hot

In [2]:
X1 = pd.get_dummies(df[['city','x']], columns=['city'])
X1.head()

## target encoding (manual, mean by category)

In [3]:
tmap = df.groupby('city')['y'].mean()
df['city_te'] = df['city'].map(tmap)
df[['city','city_te']].drop_duplicates().sort_values('city')

## downstream: how each encoding does

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
df['city_le'] = LabelEncoder().fit_transform(df['city'])
X1 = df[['x','city_le']]
X2 = pd.get_dummies(df[['x','city']], columns=['city'])
X3 = df[['x','city_te']]
y = df['y']
for name, Xi in [('label', X1), ('one-hot', X2), ('target', X3)]:
    s = cross_val_score(LogisticRegression(solver='lbfgs', max_iter=300), Xi, y, cv=5).mean()
    print(f'{name:8s}  {s:.4f}')

## ordinal mapping (manual)

In [5]:
size = pd.Series(['S','M','L','XL','S','M'])
size_o = size.map({'S':1, 'M':2, 'L':3, 'XL':4})
size_o.tolist()

In [ ]:
# re-ran with seed=42